# Engine vs. Expression
### Does the Combine engine show up on Sundays?

One physics model, the **acceleration-speed (A-S) profile**, is fitted twice for every prospect: once to his 10 Hz Combine 40-yard dash (the *engine*) and once to every frame he ran in NFL games (the *expression*). Because both live on the same axes (max acceleration **A0**, top speed **S0**), we can measure how much of a player's engine he uses, and test which part predicts NFL success **beyond what draft slot already priced in**.

Steps: (1) build the engine from Combine tracking, (2) build the expression from game tracking, (3) check test-retest reliability, (4) measure translation, (5) estimate value beyond draft slot, (6) out-of-fold incremental value.

Source code, tests and the *Prospect Translator* app: https://github.com/sanjaysaini383/NFL

## 0 · Package source (inlined so the notebook runs offline)

In [ ]:
import os, sys, tempfile
TMP = tempfile.gettempdir().replace(os.sep, '/')
SRC = f'{TMP}/c2f_src'
os.makedirs(f'{SRC}/combine2field', exist_ok=True)
sys.path.insert(0, SRC)

In [ ]:
%%writefile {SRC}/combine2field/__init__.py
"""combine2field — Engine vs. Expression.

Fit one physics model (the acceleration-speed profile) to a prospect's NFL
Combine tracking and to his in-game tracking, then measure which parts of the
Combine "engine" are actually expressed on the field and which ones predict NFL
success beyond draft slot.
"""

__version__ = "0.1.0"


In [ ]:
%%writefile {SRC}/combine2field/config.py
"""Paths, constants and position mappings."""
from __future__ import annotations

import os
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]

# Kaggle mounts competition data here; locally use data/raw (or BDB_DATA_DIR).
_KAGGLE_DIRS = [
    Path("/kaggle/input/nfl-big-data-bowl-2027"),
    Path("/kaggle/input/competitions/nfl-big-data-bowl-2027"),
    Path("/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"),
]


def on_kaggle() -> bool:
    """True only inside a Kaggle kernel (a stray local /kaggle folder must not count)."""
    return "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/input").is_dir() and os.name != "nt"


def find_kaggle_data() -> Path | None:
    """Locate players.csv anywhere under /kaggle/input (mount paths vary between kernels)."""
    for d in _KAGGLE_DIRS:
        if (d / "players.csv").exists():
            return d
    base = Path("/kaggle/input")
    if base.is_dir():
        for hit in sorted(base.rglob("players.csv")):
            if (hit.parent / "combine_tracking.csv").exists():
                return hit.parent
    return None


def data_dir() -> Path:
    env = os.environ.get("BDB_DATA_DIR")
    if env:
        return Path(env)
    if on_kaggle():
        found = find_kaggle_data()
        if found is not None:
            return found
    return ROOT / "data" / "raw"


def work_dir() -> Path:
    """Where caches and derived tables go. Kept out of git (see .gitignore)."""
    env = os.environ.get("BDB_WORK_DIR")
    if env:
        return Path(env)
    if on_kaggle():
        return Path("/kaggle/working/c2f")
    return ROOT / "outputs"


SEASONS = (2023, 2024, 2025)
LAST_SEASON = 2025
POSITION_GROUPS = ("WR", "TE", "DB", "OL", "DL")

# Snap column that defines "playing time" for each group.
SNAP_COLUMN = {
    "WR": "career_offensive_snaps",
    "TE": "career_offensive_snaps",
    "OL": "career_offensive_snaps",
    "DL": "career_defensive_snaps",
    "DB": "career_defensive_snaps",
}

# Undrafted players get a pick just past the last compensatory pick.
UNDRAFTED_PICK = 263

# Drill names in combine_tracking.csv
DRILL_FORTY = "forty_yard_dash"
DRILL_THREE_CONE = "three_cone_drill"
DRILL_SHUTTLE = "short_shuttle"
COD_DRILLS = (DRILL_THREE_CONE, DRILL_SHUTTLE)

# Tracking sample rate (Hz)
HZ = 10.0
DT = 1.0 / HZ

# In-situ A-S profile settings (Morin et al., 2021, adapted to yards).
AS_MIN_SPEED = 3.0  # yd/s, ignore the noisy low-speed region
AS_BIN = 0.2  # yd/s
AS_TOP_N = 2  # top accelerations kept per bin
AS_MIN_BINS = 8  # minimum bins required for a valid fit

# Equal exposure: every player's in-game metrics use this many random plays.
EXPOSURE_PLAYS = 80


In [ ]:
%%writefile {SRC}/combine2field/io.py
"""Loaders for the Big Data Bowl 2027 files.

Large tracking tables are converted once to Parquet in the work directory so
later runs are fast. Only the columns the pipeline needs are read.
"""
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

from . import config

NA = ["NA", ""]

COMBINE_TRACKING_COLS = [
    "nfl_id", "drill_name", "drill_type", "entity_type", "attempt", "event_id",
    "time", "x", "y", "s", "a",
]
GAME_TRACKING_COLS = ["game_id", "play_id", "nfl_id", "time", "s"]


def _path(name: str, data_dir: Path | None = None) -> Path:
    return (data_dir or config.data_dir()) / name


def _read_csv(name: str, data_dir: Path | None = None, **kw) -> pd.DataFrame:
    return pd.read_csv(_path(name, data_dir), na_values=NA, low_memory=False, **kw)


def players(data_dir: Path | None = None) -> pd.DataFrame:
    return _read_csv("players.csv", data_dir)


def combine_results(data_dir: Path | None = None) -> pd.DataFrame:
    df = _read_csv("combine_results.csv", data_dir)
    df["combine_position"] = df["combine_position"].str.upper()
    return df


def career(data_dir: Path | None = None) -> pd.DataFrame:
    return _read_csv("player_career_successes.csv", data_dir)


def games(data_dir: Path | None = None) -> pd.DataFrame:
    return _read_csv("games.csv", data_dir)


def player_play(data_dir: Path | None = None, columns: Iterable[str] | None = None) -> pd.DataFrame:
    usecols = None
    if columns is not None:
        header = pd.read_csv(_path("player_play.csv", data_dir), nrows=0).columns
        usecols = [c for c in columns if c in header]
    return _read_csv("player_play.csv", data_dir, usecols=usecols)


def time_to_seconds(t: pd.Series) -> np.ndarray:
    """Timestamps -> float seconds since epoch.

    The vectorised ISO8601 parser is ~100x faster than format="mixed", so it is
    tried first; numeric seconds are passed through.
    """
    if pd.api.types.is_numeric_dtype(t):
        return t.to_numpy(dtype=float)
    t = t.astype(str).str.replace(" ", "T", n=1, regex=False)
    try:
        ts = pd.to_datetime(t, format="ISO8601", utc=True)
    except (ValueError, TypeError):
        ts = pd.to_datetime(t, format="mixed", utc=True)
    return (ts - pd.Timestamp("2020-01-01", tz="UTC")).dt.total_seconds().to_numpy()


def _cached(name: str, usecols: list[str], data_dir: Path | None, work: Path | None,
            dtypes: dict) -> pd.DataFrame:
    work = work or config.work_dir()
    cache = work / "cache" / name.replace(".csv", ".parquet")
    if cache.exists():
        return pd.read_parquet(cache)
    src = _path(name, data_dir)
    header = pd.read_csv(src, nrows=0).columns
    cols = [c for c in usecols if c in header]
    # Read in chunks so the timestamp strings (the bulk of the memory) never all
    # sit in RAM at once; each chunk keeps only compact numeric columns.
    parts = []
    for chunk in pd.read_csv(src, usecols=cols, na_values=NA, low_memory=False, chunksize=2_000_000,
                             dtype={k: v for k, v in dtypes.items() if k in cols}):
        chunk["t"] = time_to_seconds(chunk.pop("time"))
        chunk = chunk.dropna(subset=["nfl_id"])
        chunk["nfl_id"] = chunk["nfl_id"].astype("int32")
        if "play_id" in chunk:
            chunk["play_id"] = chunk["play_id"].astype("int32")
        parts.append(chunk)
    df = pd.concat(parts, ignore_index=True)
    del parts
    cache.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(cache, index=False)
    return df


def combine_tracking(data_dir: Path | None = None, work: Path | None = None) -> pd.DataFrame:
    df = _cached(
        "combine_tracking.csv", COMBINE_TRACKING_COLS, data_dir, work,
        {"x": "float32", "y": "float32", "s": "float32", "a": "float32"},
    )
    # The real files use upper case (PLAYER, FORTY_YARD_DASH); normalise once here.
    for c in ("drill_name", "drill_type", "entity_type"):
        if c in df:
            df[c] = df[c].str.lower()
    if "entity_type" in df:
        df = df[df["entity_type"].fillna("player") == "player"]
    return df


def game_tracking(season: int, data_dir: Path | None = None, work: Path | None = None) -> pd.DataFrame:
    return _cached(
        f"game_tracking_{season}.csv", GAME_TRACKING_COLS, data_dir, work,
        {"x": "float32", "y": "float32", "s": "float32", "a": "float32"},
    )


def player_base(data_dir: Path | None = None) -> pd.DataFrame:
    """One row per player: bio, draft slot, combine results, career outcomes."""
    p = players(data_dir)
    c = combine_results(data_dir).drop(columns=["draft_year"], errors="ignore")
    k = career(data_dir)
    df = p.merge(c, on="nfl_id", how="left").merge(k, on="nfl_id", how="left")
    df = df.rename(columns={"combine_position": "pos_group"})
    df["drafted"] = df["draft_overall_pick"].notna()
    df["pick"] = df["draft_overall_pick"].fillna(config.UNDRAFTED_PICK)
    df["seasons_available"] = (config.LAST_SEASON - df["draft_year"] + 1).clip(lower=1)
    return df


In [ ]:
%%writefile {SRC}/combine2field/kinematics.py
"""Speed and tangential acceleration from 10 Hz x/y tracking.

The Combine `dis` field is unreliable, and the provided `a` is the magnitude of
acceleration, which includes centripetal acceleration in turns. For
acceleration-speed profiles we need *tangential* acceleration (dv/dt), so both
are rebuilt from the position trace.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter


def smooth(values: np.ndarray, window: int = 5, poly: int = 2) -> np.ndarray:
    if values.size < window:
        return values.astype(float)
    return savgol_filter(values.astype(float), window, poly)


def speed_from_xy(t: np.ndarray, x: np.ndarray, y: np.ndarray, window: int = 5) -> np.ndarray:
    """Speed (yd/s) from smoothed positions."""
    if t.size < 3:
        return np.zeros(t.size)
    xs, ys = smooth(x, window), smooth(y, window)
    vx = np.gradient(xs, t)
    vy = np.gradient(ys, t)
    return np.hypot(vx, vy)


def tangential_acc(t: np.ndarray, s: np.ndarray, window: int = 5) -> np.ndarray:
    if t.size < 3:
        return np.zeros(t.size)
    return np.gradient(smooth(s, window), t)


def path_length(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Cumulative distance travelled along the trace (yd)."""
    step = np.hypot(np.diff(x), np.diff(y))
    return np.concatenate([[0.0], np.cumsum(step)])


def grouped_tangential_acc(df: pd.DataFrame, keys: list[str], s_col: str = "s") -> pd.Series:
    """Centred-difference dv/dt for each tracking group, fully vectorised.

    `df` must already be sorted by keys + "t". Rows at a group boundary get NaN.
    """
    if len(df) == 0:
        return pd.Series(np.array([], dtype=float), index=df.index)
    same_prev = np.ones(len(df), dtype=bool)
    same_next = np.ones(len(df), dtype=bool)
    for k in keys:
        v = df[k].to_numpy()
        same_prev[1:] &= v[1:] == v[:-1]
        same_next[:-1] &= v[:-1] == v[1:]
    same_prev[0] = False
    same_next[-1] = False
    s = df[s_col].to_numpy(dtype=float)
    t = df["t"].to_numpy(dtype=float)
    s_next = np.roll(s, -1)
    s_prev = np.roll(s, 1)
    t_next = np.roll(t, -1)
    t_prev = np.roll(t, 1)
    ok = same_prev & same_next
    dt = np.where(ok, t_next - t_prev, np.nan)
    acc = np.where(ok & (dt > 0) & (dt < 0.35), (s_next - s_prev) / dt, np.nan)
    return pd.Series(acc, index=df.index)


In [ ]:
%%writefile {SRC}/combine2field/combine_features.py
"""The Combine "engine": sprint mechanics and change-of-direction braking.

Sprint: the 40 is fitted with the mono-exponential model used in sprint science
(Furusawa 1927; Samozino et al. 2016),

    v(t) = S0 * (1 - exp(-(t - t0) / tau)),

which implies a linear acceleration-speed relation a(v) = A0 * (1 - v / S0) with
A0 = S0 / tau. The same linear A-S relation is fitted to game tracking in
`game_profile.py`, so the Combine engine and its in-game expression sit on the
same axes.

Change of direction: the 5-10-5 shuttle and three-cone are split into turns at
local speed minima. For each turn we measure braking (peak deceleration on
entry), speed kept through the turn and re-acceleration on exit.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.signal import find_peaks

from . import config
from .kinematics import path_length, speed_from_xy, tangential_acc

ATTEMPT_KEYS = ["nfl_id", "drill_name", "attempt", "event_id"]


def _mono_exp(t, s0, tau, t0):
    return np.where(t > t0, s0 * (1 - np.exp(-(t - t0) / tau)), 0.0)


def fit_sprint(t: np.ndarray, x: np.ndarray, y: np.ndarray) -> dict:
    """Fit the sprint model to one 40-yard attempt. Returns NaNs when the trace is unusable."""
    out = {k: np.nan for k in ("S0", "tau", "A0", "Pmax_rel", "split10", "split20", "split40",
                               "vmax_obs", "fit_rmse", "fit_r2", "t0")}
    if t.size < 20:
        return out
    t = t - t[0]
    s = speed_from_xy(t, x, y)
    if s.max() < 4:
        return out
    # Start = last near-standstill frame before peak speed, so a walk-up to the
    # line before the stance does not leak into the fit.
    i_peak = int(np.argmax(s))
    still = np.flatnonzero(s[:i_peak] < 0.8)
    i0 = int(still[-1]) if still.size else 0
    d = path_length(x, y)
    d = d - d[i0]
    # fit window: from the start of motion until 42 yd or peak speed + 0.5 s
    end = min(np.searchsorted(d, 42.0), i_peak + int(0.5 * config.HZ)) + 1
    tw, sw = t[i0:end], s[i0:end]
    if tw.size < 15 or s.max() < 4:
        return out
    p0 = [float(sw.max()), 1.1, float(tw[0] + 0.2)]
    try:
        (s0, tau, t0), _ = curve_fit(
            _mono_exp, tw, sw, p0=p0,
            bounds=([3.0, 0.3, tw[0] - 0.5], [13.0, 3.0, tw[0] + 1.5]), maxfev=5000,
        )
    except (RuntimeError, ValueError):
        return out
    pred = _mono_exp(tw, s0, tau, t0)
    rss = float(np.sum((sw - pred) ** 2))
    tss = float(np.sum((sw - sw.mean()) ** 2)) or np.nan
    a0 = s0 / tau
    t_start = t0

    def split(dist):
        j = np.searchsorted(d, dist)
        return float(t[j] - t_start) if j < t.size else np.nan

    out.update({
        "S0": s0, "tau": tau, "A0": a0, "t0": t0,
        "Pmax_rel": a0 * s0 / 4.0,  # peak of a*v on the linear A-S line (yd^2/s^3, ∝ W/kg)
        "split10": split(10), "split20": split(20), "split40": split(40),
        "vmax_obs": float(s[i0:end].max()),
        "fit_rmse": np.sqrt(rss / tw.size), "fit_r2": 1 - rss / tss,
    })
    return out


def engine_from_splits(t10: float, t40: float) -> dict:
    """Recover (S0, tau, A0) from official 10- and 40-yard times.

    Solves x(t) = S0 * (t - tau * (1 - exp(-t / tau))) through both splits, so a
    prospect with only stopwatch data can be placed on the same A-S axes.
    """
    from scipy.optimize import brentq

    def dist(t, s0, tau):
        return s0 * (t - tau * (1 - np.exp(-t / tau)))

    def s0_for(tau):  # S0 that makes the 40 exact for a given tau
        return 40.0 / (t40 - tau * (1 - np.exp(-t40 / tau)))

    def resid(tau):
        return dist(t10, s0_for(tau), tau) - 10.0

    try:
        tau = brentq(resid, 0.2, min(3.0, t40 * 0.6))
    except ValueError:
        return {"S0": np.nan, "tau": np.nan, "A0": np.nan}
    s0 = s0_for(tau)
    return {"S0": s0, "tau": tau, "A0": s0 / tau}


def cod_features(t: np.ndarray, x: np.ndarray, y: np.ndarray) -> dict:
    """Braking and turn features for one change-of-direction attempt."""
    out = {k: np.nan for k in ("brake_peak", "brake_mean", "turn_speed_ratio", "reaccel_peak",
                               "drill_time", "cod_vmax", "n_turns")}
    if t.size < 15:
        return out
    t = t - t[0]
    s = speed_from_xy(t, x, y)
    a = tangential_acc(t, s)
    moving = np.flatnonzero(s > 0.8)
    if moving.size < 10:
        return out
    i0, i1 = moving[0], moving[-1]
    peak = s[i0:i1 + 1].max()
    # turns = local speed minima well below the drill's peak speed
    minima, _ = find_peaks(-s[i0:i1 + 1], prominence=0.25 * peak)
    minima = minima + i0
    brakes, ratios, reacc = [], [], []
    for m in minima:
        lo = max(i0, m - int(1.0 * config.HZ))
        hi = min(i1, m + int(1.0 * config.HZ))
        if m - lo < 2 or hi - m < 2:
            continue
        entry = s[lo:m].max()
        brakes.append(-a[lo:m + 1].min())
        reacc.append(a[m:hi + 1].max())
        if entry > 0:
            ratios.append(s[m] / entry)
    if not brakes:
        return out
    out.update({
        "brake_peak": float(np.max(brakes)),
        "brake_mean": float(np.mean(brakes)),
        "turn_speed_ratio": float(np.mean(ratios)) if ratios else np.nan,
        "reaccel_peak": float(np.mean(reacc)),
        "drill_time": float(t[i1] - t[i0]),
        "cod_vmax": float(peak),
        "n_turns": len(brakes),
    })
    return out


def attempt_features(ct: pd.DataFrame) -> pd.DataFrame:
    """One row per Combine attempt (40s and COD drills) with engine features."""
    keys = [k for k in ATTEMPT_KEYS if k in ct.columns]
    drills = [config.DRILL_FORTY, *config.COD_DRILLS]
    ct = ct[ct["drill_name"].isin(drills)].sort_values(keys + ["t"])
    rows = []
    for key, g in ct.groupby(keys, sort=False):
        rec = dict(zip(keys, key))
        t = g["t"].to_numpy(float)
        x = g["x"].to_numpy(float)
        y = g["y"].to_numpy(float)
        if rec["drill_name"] == config.DRILL_FORTY:
            rec.update(fit_sprint(t, x, y))
        else:
            rec.update(cod_features(t, x, y))
        rows.append(rec)
    return pd.DataFrame(rows)


SPRINT_COLS = ["S0", "tau", "A0", "Pmax_rel", "split10", "split20", "split40", "vmax_obs"]
COD_COLS = ["brake_peak", "brake_mean", "turn_speed_ratio", "reaccel_peak", "drill_time"]


def player_engine(attempts: pd.DataFrame, min_r2: float = 0.9) -> pd.DataFrame:
    """Collapse attempts to one row per player.

    Sprint: best attempt (fastest observed split40 among well-fitted attempts),
    mirroring how the official Combine keeps the best time.
    COD: braking/turn features are averaged per drill, then over drills.
    """
    out = []
    forty = attempts[attempts["drill_name"] == config.DRILL_FORTY]
    if not forty.empty:
        good = forty[forty["fit_r2"] >= min_r2].copy()
        good["_rank"] = good["split40"].fillna(99)
        best = good.sort_values("_rank").groupby("nfl_id").head(1)
        out.append(best.set_index("nfl_id")[SPRINT_COLS + ["fit_r2"]].add_prefix("eng_"))
        out.append(good.groupby("nfl_id").size().rename("eng_n_forty"))
    cod = attempts[attempts["drill_name"].isin(config.COD_DRILLS)]
    if not cod.empty:
        per_drill = cod.groupby(["nfl_id", "drill_name"])[COD_COLS].mean()
        wide = per_drill.drop(columns="drill_time").groupby("nfl_id").mean().add_prefix("eng_")
        times = per_drill["drill_time"].unstack("drill_name").add_prefix("eng_time_")
        out += [wide, times]
    if not out:
        return pd.DataFrame(index=pd.Index([], name="nfl_id"))
    return pd.concat(out, axis=1).rename_axis("nfl_id").reset_index()


def retest_pairs(attempts: pd.DataFrame, cols: list[str] | None = None) -> pd.DataFrame:
    """Attempt 1 vs attempt 2 for each player/drill, in long form, for reliability."""
    cols = cols or SPRINT_COLS + COD_COLS
    rows = []
    for (pid, drill), g in attempts.groupby(["nfl_id", "drill_name"]):
        g = g.sort_values("attempt")
        if len(g) < 2:
            continue
        a, b = g.iloc[0], g.iloc[1]
        for c in cols:
            if c in g and pd.notna(a.get(c)) and pd.notna(b.get(c)):
                rows.append({"nfl_id": pid, "drill_name": drill, "metric": c, "trial1": a[c], "trial2": b[c]})
    return pd.DataFrame(rows)


In [ ]:
%%writefile {SRC}/combine2field/game_profile.py
"""In-game "expression": in-situ acceleration-speed profiles from game tracking.

Method (Morin et al., 2021, "Individual acceleration-speed profile in-situ"):
pool every frame a player produced, bin speed in 0.2 yd/s steps above 3 yd/s,
keep the two highest tangential accelerations per bin, and fit a straight line
a = A0 - (A0/S0) * v. Points outside the 95% prediction band are dropped and the
line is refitted. The intercept A0 and the speed intercept S0 are directly
comparable to the Combine sprint fit.

The game files have ~22.6M rows, so the reduction runs per season and keeps only
the per-bin top accelerations and per-play peaks for each player.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from . import config, io
from .kinematics import grouped_tangential_acc

PLAY_KEYS = ["nfl_id", "game_id", "play_id"]
MAX_SPEED = 12.5  # yd/s; anything faster is a tracking glitch
MAX_ACC = 12.0  # yd/s^2


def frame_reduce(gt: pd.DataFrame, nfl_ids: set | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Reduce one season of game tracking.

    Returns
    -------
    bins : nfl_id, bin, a   (top-N tangential accelerations per 0.2 yd/s bin)
    plays: nfl_id, game_id, play_id, s_max, a_min, a_max  (per-play peaks)
    """
    if nfl_ids is not None:
        gt = gt[gt["nfl_id"].isin(nfl_ids)]
    gt = gt[PLAY_KEYS + ["t", "s"]].dropna().sort_values(PLAY_KEYS + ["t"], kind="stable",
                                                         ignore_index=True)
    gt["a_tan"] = grouped_tangential_acc(gt, PLAY_KEYS).to_numpy(dtype="float32")
    gt = gt[(gt["s"] <= MAX_SPEED) & (gt["a_tan"].abs() <= MAX_ACC)]

    plays = gt.groupby(PLAY_KEYS, sort=False).agg(
        s_max=("s", "max"), a_min=("a_tan", "min"), a_max=("a_tan", "max"),
    ).reset_index()

    hs = gt[gt["s"] >= config.AS_MIN_SPEED]
    b = np.floor((hs["s"].to_numpy() - config.AS_MIN_SPEED) / config.AS_BIN).astype(int)
    bins = pd.DataFrame({"nfl_id": hs["nfl_id"].to_numpy(), "bin": b, "a": hs["a_tan"].to_numpy()})
    bins = top_n_per_bin(bins)
    return bins, plays


def top_n_per_bin(bins: pd.DataFrame, n: int = config.AS_TOP_N) -> pd.DataFrame:
    bins = bins.sort_values(["nfl_id", "bin", "a"], ascending=[True, True, False])
    return bins.groupby(["nfl_id", "bin"], sort=False).head(n).reset_index(drop=True)


def fit_as_profile(speed: np.ndarray, acc: np.ndarray) -> dict:
    """Robust linear A-S fit. Returns A0 (yd/s^2), S0 (yd/s), r2, n points."""
    out = {"A0": np.nan, "S0": np.nan, "as_r2": np.nan, "as_n": int(speed.size)}
    ok = (acc > 0) & np.isfinite(acc)
    v, a = speed[ok], acc[ok]
    if v.size < 2 * config.AS_MIN_BINS:
        return out
    for _ in range(2):
        slope, inter, r, _, _ = stats.linregress(v, a)
        resid = a - (inter + slope * v)
        sd = resid.std(ddof=2) if v.size > 3 else 0
        keep = np.abs(resid) <= 1.96 * sd if sd > 0 else np.ones_like(v, bool)
        if keep.all():
            break
        v, a = v[keep], a[keep]
        if v.size < 2 * config.AS_MIN_BINS:
            return out
    slope, inter, r, _, _ = stats.linregress(v, a)
    if slope >= 0 or inter <= 0:
        return out
    out.update({"A0": inter, "S0": -inter / slope, "as_r2": r ** 2, "as_n": int(v.size)})
    return out


def player_expression(bins: pd.DataFrame, plays: pd.DataFrame) -> pd.DataFrame:
    """One row per player: in-game A-S profile and peak-intensity summaries."""
    bins = top_n_per_bin(bins)
    rows = []
    for pid, g in bins.groupby("nfl_id"):
        v = config.AS_MIN_SPEED + (g["bin"].to_numpy() + 0.5) * config.AS_BIN
        rows.append({"nfl_id": pid, **fit_as_profile(v, g["a"].to_numpy())})
    prof = pd.DataFrame(rows).set_index("nfl_id").add_prefix("game_") if rows else pd.DataFrame()
    peaks = plays.groupby("nfl_id").agg(
        game_n_plays=("s_max", "size"),
        game_vmax=("s_max", lambda x: np.nanpercentile(x, 99.5)),
        game_v95=("s_max", lambda x: np.nanpercentile(x, 95)),
        game_brake=("a_min", lambda x: -np.nanpercentile(x, 1)),
        game_accel=("a_max", lambda x: np.nanpercentile(x, 99)),
    )
    return peaks.join(prof, how="left").rename_axis("nfl_id").reset_index()


def sample_plays(play_keys: pd.DataFrame, k: int, seed: int = 2027) -> pd.DataFrame:
    """Pick the same number of plays (k) for every player, at random.

    Peak-based metrics rise with the number of chances a player gets, and
    players who earn more snaps get more chances. Measuring everyone on k
    random plays removes that exposure effect. Players with fewer than k
    tracked plays are dropped rather than measured on a smaller sample.
    """
    rng = np.random.default_rng(seed)
    pk = play_keys.drop_duplicates().assign(_u=lambda d: rng.random(len(d)))
    pk["_rank"] = pk.groupby("nfl_id")["_u"].rank(method="first")
    n = pk.groupby("nfl_id")["_u"].transform("size")
    return pk[(pk["_rank"] <= k) & (n >= k)][PLAY_KEYS].reset_index(drop=True)


def build(nfl_ids: set | None = None, data_dir: Path | None = None, work: Path | None = None,
          seasons=config.SEASONS, plays_per_player: int | None = config.EXPOSURE_PLAYS,
          ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run the per-season reduction and return (expression, bins, plays).

    With `plays_per_player` set, every player is measured on the same number of
    randomly sampled plays (equal exposure); `game_n_plays_total` keeps the full count.
    """
    def load(season):
        try:
            gt = io.game_tracking(season, data_dir, work)
        except FileNotFoundError:
            return None
        return gt[gt["nfl_id"].isin(nfl_ids)] if nfl_ids is not None else gt

    # pass 1: which plays exist for each player
    keys = [g[PLAY_KEYS].drop_duplicates() for s in seasons if (g := load(s)) is not None]
    if not keys:
        raise FileNotFoundError("No game_tracking_<season>.csv files found")
    keys = pd.concat(keys, ignore_index=True)
    totals = keys.groupby("nfl_id").size().rename("game_n_plays_total")
    chosen = sample_plays(keys, plays_per_player) if plays_per_player else keys
    if chosen.empty:
        raise ValueError(f"No player has {plays_per_player} tracked plays; lower plays_per_player")
    chosen_idx = pd.MultiIndex.from_frame(chosen)

    # pass 2: reduce only the chosen plays
    all_bins, all_plays = [], []
    for season in seasons:
        gt = load(season)
        if gt is None:
            continue
        gt = gt[pd.MultiIndex.from_frame(gt[PLAY_KEYS]).isin(chosen_idx)]
        b, p = frame_reduce(gt)
        all_bins.append(b)
        all_plays.append(p)
        del gt
    bins = pd.concat(all_bins, ignore_index=True)
    plays = pd.concat(all_plays, ignore_index=True)
    expr = player_expression(bins, plays).merge(totals.reset_index(), on="nfl_id", how="right")
    return expr, top_n_per_bin(bins), plays


In [ ]:
%%writefile {SRC}/combine2field/outcomes.py
"""Position-specific definitions of "NFL success".

Career outcomes are normalised by the seasons a player has been in the league
(the 2023 class has three seasons, the 2025 class one). Play-level outcomes come
from player_play.csv and are only computed with enough plays to be stable.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import config

PLAY_COLUMNS = [
    "game_id", "play_id", "nfl_id", "route_ran", "separation_at_pass_forward", "target",
    "pressure_allowed", "sack_allowed", "time_to_pressure", "player_get_off",
    "pass_rushers_encountered", "tackle", "assist", "sack",
]
MIN_PLAYS = 30

OUTCOME_LABELS = {
    "snaps_per_season": "Snaps per season",
    "starts_per_season": "Starts per season",
    "starter": "Became a starter (≥8 starts/season)",
    "sep_mean": "Avg separation at throw (yd)",
    "target_rate": "Targets per route",
    "pressure_allowed_rate": "Pressure allowed rate (lower = better)",
    "pressure_rate": "Pressure rate per pass rush",
    "get_off": "Get-off time (s, lower = better)",
    "tackle_rate": "Tackles per play",
}
# outcomes where lower is better; their sign is flipped in analysis
LOWER_IS_BETTER = {"pressure_allowed_rate", "get_off"}

PRIMARY_PLAY_OUTCOME = {
    "WR": "sep_mean", "TE": "sep_mean", "OL": "pressure_allowed_rate",
    "DL": "pressure_rate", "DB": "tackle_rate",
}


def _bool(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.astype(float)
    m = {"true": 1.0, "false": 0.0, "1": 1.0, "0": 0.0, "1.0": 1.0, "0.0": 0.0}
    return s.astype(str).str.lower().map(m)


def career_outcomes(base: pd.DataFrame) -> pd.DataFrame:
    seasons = base["seasons_available"].astype(float)
    snaps = np.select(
        [base["pos_group"].map(config.SNAP_COLUMN) == c for c in
         ("career_offensive_snaps", "career_defensive_snaps")],
        [base["career_offensive_snaps"], base["career_defensive_snaps"]],
        default=np.nan,
    )
    out = pd.DataFrame({"nfl_id": base["nfl_id"]})
    out["snaps_per_season"] = snaps / seasons
    out["starts_per_season"] = base["career_games_started"] / seasons
    out["starter"] = (out["starts_per_season"] >= 8).astype(float)
    return out


def play_outcomes(pp: pd.DataFrame) -> pd.DataFrame:
    pp = pp.copy()
    for c in ("target", "pressure_allowed", "sack_allowed"):
        if c in pp:
            pp[c] = _bool(pp[c])
    parts = []
    if "route_ran" in pp:
        routes = pp[pp["route_ran"].notna()]
        r = routes.groupby("nfl_id").agg(
            n_routes=("play_id", "size"),
            sep_mean=("separation_at_pass_forward", "mean"),
            target_rate=("target", "mean"),
        )
        r.loc[r["n_routes"] < MIN_PLAYS, ["sep_mean", "target_rate"]] = np.nan
        parts.append(r)
    if "pressure_allowed" in pp:
        pb = pp[pp["pressure_allowed"].notna()]
        b = pb.groupby("nfl_id").agg(n_pass_block=("play_id", "size"),
                                     pressure_allowed_rate=("pressure_allowed", "mean"))
        b.loc[b["n_pass_block"] < MIN_PLAYS, "pressure_allowed_rate"] = np.nan
        parts.append(b)
    if "player_get_off" in pp:
        rush = pp[pp["player_get_off"].notna()].assign(
            pressured=lambda d: d["time_to_pressure"].notna().astype(float))
        k = rush.groupby("nfl_id").agg(n_pass_rush=("play_id", "size"),
                                       pressure_rate=("pressured", "mean"),
                                       get_off=("player_get_off", "mean"))
        k.loc[k["n_pass_rush"] < MIN_PLAYS, ["pressure_rate", "get_off"]] = np.nan
        parts.append(k)
    if "tackle" in pp:
        # tackle/assist are only filled when the player was involved; blank = no tackle.
        # Solo tackles count 1, assists 0.5.
        pp["tackle_credit"] = (pp["tackle"].fillna(0) == 1).astype(float)
        if "assist" in pp:
            pp["tackle_credit"] += 0.5 * (pp["assist"].fillna(0) == 1)
        t = pp.groupby("nfl_id").agg(n_plays=("play_id", "size"), tackle_rate=("tackle_credit", "mean"))
        t.loc[t["n_plays"] < MIN_PLAYS, "tackle_rate"] = np.nan
        parts.append(t)
    if not parts:
        return pd.DataFrame(columns=["nfl_id"])
    return pd.concat(parts, axis=1).rename_axis("nfl_id").reset_index()


In [ ]:
%%writefile {SRC}/combine2field/stats.py
"""Small, dependency-light statistics helpers.

With roughly 100 players per position and many candidate metrics, the analysis
leans on rank statistics, bootstrap intervals, false-discovery control and
out-of-fold prediction rather than in-sample fits.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def _clean(*arrays):
    a = np.column_stack([np.asarray(x, float) for x in arrays])
    a = a[np.isfinite(a).all(axis=1)]
    return [a[:, i] for i in range(a.shape[1])]


def spearman_ci(x, y, n_boot: int = 1000, seed: int = 0) -> dict:
    x, y = _clean(x, y)
    n = x.size
    if n < 8 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return {"n": n, "rho": np.nan, "lo": np.nan, "hi": np.nan, "p": np.nan}
    rho, p = stats.spearmanr(x, y)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, (n_boot, n))
    rx, ry = stats.rankdata(x), stats.rankdata(y)
    boots = []
    for i in idx:
        a, b = rx[i], ry[i]
        if a.std() > 0 and b.std() > 0:
            boots.append(np.corrcoef(a, b)[0, 1])
    lo, hi = np.percentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan)
    return {"n": n, "rho": rho, "lo": lo, "hi": hi, "p": p}


def partial_spearman(x, y, z) -> dict:
    """Spearman correlation of x and y after removing the rank-linear effect of z."""
    x, y, z = _clean(x, y, z)
    n = x.size
    if n < 10:
        return {"prho": np.nan, "pp": np.nan}
    rx, ry, rz = (stats.rankdata(v) for v in (x, y, z))
    Z = np.column_stack([np.ones(n), rz])
    ex = rx - Z @ np.linalg.lstsq(Z, rx, rcond=None)[0]
    ey = ry - Z @ np.linalg.lstsq(Z, ry, rcond=None)[0]
    if ex.std() == 0 or ey.std() == 0:
        return {"prho": np.nan, "pp": np.nan}
    r = np.corrcoef(ex, ey)[0, 1]
    df = n - 3
    t = r * np.sqrt(df / max(1 - r ** 2, 1e-12))
    return {"prho": r, "pp": 2 * stats.t.sf(abs(t), df)}


def partial_spearman_ci(x, y, z, n_boot: int = 2000, seed: int = 0) -> dict:
    """Partial Spearman (controlling for z) with a percentile bootstrap interval."""
    x, y, z = _clean(x, y, z)
    point = partial_spearman(x, y, z)
    n = x.size
    if n < 15 or not np.isfinite(point["prho"]):
        return {**point, "n": n, "plo": np.nan, "phi": np.nan}
    rng = np.random.default_rng(seed)
    boots = [partial_spearman(x[i], y[i], z[i])["prho"] for i in rng.integers(0, n, (n_boot, n))]
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return {**point, "n": n, "plo": lo, "phi": hi}


def bh_qvalues(p) -> np.ndarray:
    """Benjamini-Hochberg adjusted p-values (NaNs preserved)."""
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    if not ok.any():
        return q
    pv = p[ok]
    order = np.argsort(pv)
    m = pv.size
    ranked = pv[order] * m / np.arange(1, m + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.clip(ranked, 0, 1)
    q[ok] = out
    return q


def icc_3_1(t1, t2) -> float:
    """ICC(3,1): two-way mixed, consistency, single measurement (Shrout & Fleiss)."""
    t1, t2 = _clean(t1, t2)
    n = t1.size
    if n < 5:
        return np.nan
    Y = np.column_stack([t1, t2])
    k = 2
    grand = Y.mean()
    ss_rows = k * ((Y.mean(axis=1) - grand) ** 2).sum()
    ss_cols = n * ((Y.mean(axis=0) - grand) ** 2).sum()
    ss_err = ((Y - grand) ** 2).sum() - ss_rows - ss_cols
    ms_rows = ss_rows / (n - 1)
    ms_err = ss_err / ((n - 1) * (k - 1))
    denom = ms_rows + (k - 1) * ms_err
    return float((ms_rows - ms_err) / denom) if denom > 0 else np.nan


def cv_r2(df: pd.DataFrame, y: str, features: list[str], n_repeats: int = 20, seed: int = 0) -> np.ndarray:
    """Out-of-fold R^2 on the rank-transformed outcome, one value per CV repeat."""
    d = df[df[y].notna()]
    feats = [f for f in features if f in d and d[f].notna().any()]
    if len(d) < 25 or not feats:
        return np.array([np.nan])
    X = d[feats].to_numpy(float)
    target = stats.rankdata(d[y].to_numpy(float)) / len(d)
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          RidgeCV(alphas=np.logspace(-2, 3, 20)))
    scores = []
    rkf = RepeatedKFold(n_splits=5, n_repeats=n_repeats, random_state=seed)
    pred = np.zeros_like(target)
    for i, (tr, te) in enumerate(rkf.split(X)):
        model.fit(X[tr], target[tr])
        pred[te] = model.predict(X[te])
        if (i + 1) % 5 == 0:
            ss_res = ((target - pred) ** 2).sum()
            ss_tot = ((target - target.mean()) ** 2).sum()
            scores.append(1 - ss_res / ss_tot)
    return np.array(scores)


In [ ]:
%%writefile {SRC}/combine2field/translation.py
"""Engine vs. Expression: put Combine and game physics side by side and test them.

Questions answered, per position group:
1. Reliability: which tracking metrics repeat across Combine attempts (ICC)?
2. Translation: does the Combine engine (A0, S0, braking) show up in games?
3. Value: which metrics predict NFL success *beyond draft slot*?
4. Increment: do tracking metrics add out-of-fold predictive value on top of
   draft slot and the traditional stopwatch Combine?
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import config, stats
from .outcomes import LOWER_IS_BETTER, PRIMARY_PLAY_OUTCOME

ENGINE_FEATURES = {
    "eng_S0": "Combine top speed S0 (yd/s)",
    "eng_A0": "Combine max accel A0 (yd/s²)",
    "eng_tau": "Shorter accel time constant τ",
    "eng_Pmax_rel": "Combine relative power",
    "eng_brake_peak": "Combine peak braking (yd/s²)",
    "eng_turn_speed_ratio": "Combine turn speed retention",
    "eng_reaccel_peak": "Combine re-acceleration (yd/s²)",
}
EXPRESSION_FEATURES = {
    "game_S0": "Game A-S top speed S0 (yd/s)",
    "game_A0": "Game A-S max accel A0 (yd/s²)",
    "game_vmax": "Game peak speed (yd/s)",
    "game_brake": "Game peak braking (yd/s²)",
}
RATIO_FEATURES = {
    "expr_speed": "Speed expression (game peak / Combine S0)",
    "expr_accel": "Accel expression (game A0 / Combine A0)",
    "expr_brake": "Braking expression (game / Combine)",
}
# Finer roles inside a position group (from players.nfl_position).
ROLES = {"DE": "EDGE", "OLB": "EDGE", "DT": "IDL", "NT": "IDL"}
PASS_RUSH_FEATURES = ["eng_Pmax_rel", "ten_yd_split", "eng_A0", "eng_S0", "forty",
                      "game_A0", "game_brake", "game_vmax", "combine_weight"]

TRADITIONAL = ["forty", "ten_yd_split", "vertical", "broad_jump", "three_cone", "short_shuttle",
               "bench_reps", "combine_weight", "combine_height", "arm_length"]
FEATURE_LABELS = {**ENGINE_FEATURES, **EXPRESSION_FEATURES, **RATIO_FEATURES,
                  "forty": "Faster 40 (official)", "ten_yd_split": "Faster 10-yd split (official)",
                  "combine_weight": "Lighter body weight", "ngs_athleticism_score": "NGS athleticism score"}
# metrics where a smaller value means a better athlete
SMALLER_IS_BETTER = {"eng_tau", "forty", "ten_yd_split", "three_cone", "short_shuttle", "combine_weight"}


def assemble(base: pd.DataFrame, engine: pd.DataFrame, expression: pd.DataFrame,
             career: pd.DataFrame, plays: pd.DataFrame) -> pd.DataFrame:
    df = (base.merge(engine, on="nfl_id", how="left")
              .merge(expression, on="nfl_id", how="left")
              .merge(career, on="nfl_id", how="left")
              .merge(plays, on="nfl_id", how="left"))
    df["expr_speed"] = df["game_vmax"] / df["eng_S0"]
    df["expr_accel"] = df["game_A0"] / df["eng_A0"]
    df["expr_brake"] = df["game_brake"] / df["eng_brake_peak"]
    df["log_pick"] = np.log(df["pick"])
    df["role"] = df["nfl_position"].astype(str).str.upper().map(ROLES).fillna(df["pos_group"])
    # position percentiles make cards and comps readable
    for c in list(ENGINE_FEATURES) + list(EXPRESSION_FEATURES) + list(RATIO_FEATURES):
        if c in df:
            df[f"pct_{c}"] = df.groupby("pos_group")[c].rank(
                pct=True, ascending=c not in SMALLER_IS_BETTER)
    return df


def reliability(pairs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (drill, metric), g in pairs.groupby(["drill_name", "metric"]):
        rows.append({"drill_name": drill, "metric": metric, "n": len(g),
                     "icc": stats.icc_3_1(g["trial1"], g["trial2"]),
                     "r": g[["trial1", "trial2"]].corr().iloc[0, 1] if len(g) > 2 else np.nan})
    return pd.DataFrame(rows).sort_values("icc", ascending=False)


def translation_table(df: pd.DataFrame) -> pd.DataFrame:
    """Combine engine vs in-game expression of the same quantity."""
    pairs = [("eng_S0", "game_vmax"), ("eng_S0", "game_S0"), ("eng_A0", "game_A0"),
             ("eng_brake_peak", "game_brake"), ("forty", "game_vmax")]
    rows = []
    for pos, g in df.groupby("pos_group"):
        for a, b in pairs:
            if a in g and b in g:
                r = stats.spearman_ci(g[a], g[b])
                rows.append({"pos_group": pos, "combine": a, "game": b, **r})
    return pd.DataFrame(rows)


def _outcomes_for(pos: str) -> list[str]:
    return ["snaps_per_season", "starts_per_season", PRIMARY_PLAY_OUTCOME.get(pos)]


def association_table(df: pd.DataFrame, features: list[str] | None = None) -> pd.DataFrame:
    """Spearman and draft-adjusted partial Spearman for every feature x outcome x position.

    Signs are oriented so that a positive value always means "better athlete ->
    better outcome". q-values use Benjamini-Hochberg within each position.
    """
    features = features or [f for f in FEATURE_LABELS if f in df]
    rows = []
    for pos, g in df.groupby("pos_group"):
        for out in _outcomes_for(pos):
            if out is None or out not in g or g[out].notna().sum() < 10:
                continue
            y = g[out] * (-1 if out in LOWER_IS_BETTER else 1)
            for f in features:
                if f not in g:
                    continue
                x = g[f] * (-1 if f in SMALLER_IS_BETTER else 1)
                r = stats.spearman_ci(x, y)
                pr = stats.partial_spearman(x, y, g["log_pick"])
                rows.append({"pos_group": pos, "outcome": out, "feature": f, **r, **pr})
    res = pd.DataFrame(rows)
    if res.empty:
        return res
    res["q"] = res.groupby("pos_group")["p"].transform(stats.bh_qvalues)
    res["pq"] = res.groupby("pos_group")["pp"].transform(stats.bh_qvalues)
    return res


def pass_rush_table(df: pd.DataFrame, n_boot: int = 2000) -> pd.DataFrame:
    """DL pressure rate vs each trait, draft-adjusted, split into EDGE and interior (IDL).

    Edge rushers are lighter, faster and pressure more often, so a pooled DL
    correlation partly measures role. Splitting by role removes that.
    """
    dl = df[df["pos_group"] == "DL"]
    rows = []
    for role, g in [("All DL", dl), *dl.groupby("role")]:
        for f in PASS_RUSH_FEATURES:
            if f not in g:
                continue
            x = g[f] * (-1 if f in SMALLER_IS_BETTER else 1)
            r = stats.partial_spearman_ci(x, g["pressure_rate"], g["log_pick"], n_boot=n_boot)
            rows.append({"role": role, "feature": f, **r})
    out = pd.DataFrame(rows)
    if not out.empty:
        out["pq"] = out.groupby("role")["pp"].transform(stats.bh_qvalues)
    return out


def incremental_value(df: pd.DataFrame, n_repeats: int = 20) -> pd.DataFrame:
    """Out-of-fold R^2 for nested models: draft slot -> + stopwatch Combine -> + tracking."""
    tracking = ["eng_S0", "eng_A0", "eng_brake_peak", "eng_turn_speed_ratio",
                "game_S0", "game_A0", "game_vmax", "expr_speed"]
    models = {
        "Draft slot": ["log_pick"],
        "+ Stopwatch Combine": ["log_pick", *TRADITIONAL],
        "+ Tracking (engine & expression)": ["log_pick", *TRADITIONAL, *tracking],
    }
    rows = []
    for pos, g in df.groupby("pos_group"):
        for out in _outcomes_for(pos):
            if out is None or out not in g:
                continue
            for name, feats in models.items():
                sc = stats.cv_r2(g, out, feats, n_repeats=n_repeats)
                rows.append({"pos_group": pos, "outcome": out, "model": name,
                             "r2": np.nanmean(sc), "r2_sd": np.nanstd(sc), "n": int(g[out].notna().sum())})
    return pd.DataFrame(rows)


def comps(df: pd.DataFrame, nfl_id: int, k: int = 5,
          features=("pct_eng_S0", "pct_eng_A0", "pct_eng_brake_peak", "pct_game_vmax", "pct_game_A0")) -> pd.DataFrame:
    """Nearest neighbours within a position group on percentile profile."""
    me = df.loc[df["nfl_id"] == nfl_id].iloc[0]
    pool = df[(df["pos_group"] == me["pos_group"]) & (df["nfl_id"] != nfl_id)]
    feats = [f for f in features if f in df and pd.notna(me.get(f))]
    if not feats:
        return pool.head(0)
    d = np.sqrt(((pool[feats] - me[feats].astype(float)) ** 2).sum(axis=1, min_count=1))
    return pool.assign(distance=d).nsmallest(k, "distance")


In [ ]:
%%writefile {SRC}/combine2field/report.py
"""Static figures for the write-up (matplotlib, light theme for Kaggle)."""
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402

from . import config  # noqa: E402
from .combine_features import _mono_exp, fit_sprint  # noqa: E402
from .kinematics import speed_from_xy  # noqa: E402
from .outcomes import PRIMARY_PLAY_OUTCOME  # noqa: E402
from .translation import FEATURE_LABELS  # noqa: E402

METRIC_LABELS = {
    "vmax_obs": "Peak speed", "S0": "Top speed S0", "split40": "40-yd split", "split20": "20-yd split",
    "split10": "10-yd split", "Pmax_rel": "Relative power", "A0": "Max accel A0", "tau": "Time constant τ",
    "drill_time": "Drill time", "turn_speed_ratio": "Turn speed retention", "brake_mean": "Mean braking",
    "brake_peak": "Peak braking", "reaccel_peak": "Re-acceleration",
}
DRILL_LABELS = {"forty_yard_dash": "40", "three_cone_drill": "3-cone", "short_shuttle": "shuttle"}

# Reference palette (dataviz skill): surfaces, ink, categorical slots, diverging.
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]  # first three slots validate all-pairs
NEUTRAL = "#f0efec"
DIVERGING = LinearSegmentedColormap.from_list(
    "c2f_div", ["#b7302f", "#e34948", "#f4a3a2", NEUTRAL, "#9ec5f4", "#2a78d6", "#184f95"])
POS_ORDER = list(config.POSITION_GROUPS)


def _font() -> list[str]:
    """First installed UI font, so Linux (Kaggle) doesn't log a warning per text element."""
    from matplotlib import font_manager
    have = {f.name for f in font_manager.fontManager.ttflist}
    return [f for f in ("Segoe UI", "DejaVu Sans") if f in have] or ["sans-serif"]


def _style():
    plt.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "text.color": INK,
        "ytick.labelcolor": INK2, "xtick.labelcolor": INK2,
        "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
        "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
        "font.family": _font(), "font.size": 10,
        "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "lines.linewidth": 2, "legend.frameon": False,
    })


def _save(fig, out: Path, name: str) -> Path:
    out.mkdir(parents=True, exist_ok=True)
    p = out / f"{name}.png"
    fig.savefig(p, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return p


def fig_method(ct: pd.DataFrame, bins: pd.DataFrame, player: pd.Series, out: Path) -> Path:
    """One player, one model: Combine 40 speed curve and in-game A-S cloud on shared axes."""
    _style()
    pid = player["nfl_id"]
    forty = ct[(ct["nfl_id"] == pid) & (ct["drill_name"] == config.DRILL_FORTY)]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    if not forty.empty:
        att = forty.groupby("attempt")
        g = att.get_group(sorted(att.groups)[0]).sort_values("t")
        t = g["t"].to_numpy() - g["t"].iloc[0]
        x, y = g["x"].to_numpy(float), g["y"].to_numpy(float)
        s = speed_from_xy(t, x, y)
        fit = fit_sprint(t, x, y)
        a1.plot(t, s, color=MUTED, lw=1.2, marker="o", ms=2.5, label="Tracked speed (10 Hz)")
        if np.isfinite(fit["S0"]):
            tt = np.linspace(0, t.max(), 300)
            a1.plot(tt, _mono_exp(tt, fit["S0"], fit["tau"], fit["t0"]), color=SERIES[0],
                    label=f"Fit: S0={fit['S0']:.1f} yd/s, τ={fit['tau']:.2f} s")
    a1.set(title="Combine 40: the engine", xlabel="Time (s)", ylabel="Speed (yd/s)")
    a1.legend(loc="lower right", fontsize=8)

    b = bins[bins["nfl_id"] == pid]
    v = config.AS_MIN_SPEED + (b["bin"] + 0.5) * config.AS_BIN
    a2.scatter(v, b["a"], s=14, color=SERIES[1], alpha=0.75, label="Game frames (top-2 per bin)",
               edgecolor=SURFACE, linewidth=0.6)
    vv = np.linspace(0, max(player.get("eng_S0", 10) or 10, 1) * 1.02, 50)
    if pd.notna(player.get("game_A0")):
        a2.plot(vv, np.clip(player["game_A0"] * (1 - vv / player["game_S0"]), 0, None),
                color=SERIES[1], label=f"Game A-S: A0={player['game_A0']:.1f}, S0={player['game_S0']:.1f}")
    if pd.notna(player.get("eng_A0")):
        a2.plot(vv, np.clip(player["eng_A0"] * (1 - vv / player["eng_S0"]), 0, None),
                color=SERIES[0], ls="--", label=f"Combine A-S: A0={player['eng_A0']:.1f}, S0={player['eng_S0']:.1f}")
    a2.set(title="Games: the expression", xlabel="Speed (yd/s)", ylabel="Acceleration (yd/s²)")
    a2.set_ylim(bottom=0)
    a2.legend(loc="upper right", fontsize=8)
    fig.suptitle(f"{player.get('display_name', pid)} ({player.get('pos_group', '')}): "
                 "one acceleration-speed model, two environments", x=0.01, ha="left",
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    return _save(fig, out, "fig1_method")


def fig_reliability(rel: pd.DataFrame, out: Path, top: int = 14) -> Path:
    _style()
    r = rel.dropna(subset=["icc"]).head(top).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 0.32 * len(r) + 1.2))
    labels = [f"{METRIC_LABELS.get(m, m)}  ·  {DRILL_LABELS.get(d, d)}"
              for d, m in zip(r["drill_name"], r["metric"])]
    ax.barh(labels, r["icc"], color=SERIES[0], height=0.62)
    for i, v in enumerate(r["icc"]):
        ax.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=8, color=INK2)
    ax.axvline(0.75, color=MUTED, lw=1, ls=":")
    ax.text(0.75, len(r) - 0.3, " good (0.75)", color=MUTED, fontsize=8)
    ax.set(xlim=(0, 1.08), xlabel="Test-retest ICC(3,1), attempt 1 vs 2",
           title="Which tracking metrics repeat across Combine attempts?")
    ax.grid(axis="y", visible=False)
    return _save(fig, out, "fig2_reliability")


def fig_translation(df: pd.DataFrame, out: Path, x="eng_S0", y="game_vmax") -> Path:
    _style()
    pos = [p for p in POS_ORDER if p in set(df["pos_group"])]
    fig, axes = plt.subplots(1, len(pos), figsize=(2.6 * len(pos), 3), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    d = df.dropna(subset=[x, y])
    lo, hi = (d[[x, y]].min().min() * 0.95, d[[x, y]].max().max() * 1.03) if len(d) else (0, 1)
    for ax, p in zip(axes, pos):
        g = d[d["pos_group"] == p]
        ax.plot([lo, hi], [lo, hi], color=AXIS, lw=1, ls="--")
        ax.scatter(g[x], g[y], s=16, color=SERIES[0], alpha=0.8, edgecolor=SURFACE, linewidth=0.6)
        med = (g[y] / g[x]).median() if len(g) else np.nan
        ax.set_title(f"{p}  ·  {med:.0%} expressed" if np.isfinite(med) else p)
        ax.set_xlabel("Combine S0 (yd/s)")
    axes[0].set_ylabel("Game peak speed (yd/s)")
    fig.suptitle("How much of the Combine top speed shows up in games? (dashed = 100%)",
                 x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.tight_layout()
    return _save(fig, out, "fig3_translation")


def fig_heatmap(assoc: pd.DataFrame, out: Path, outcome="snaps_per_season", value="prho") -> Path:
    _style()
    if outcome == "primary":  # each position's own play-level outcome
        a = assoc[assoc["outcome"] == assoc["pos_group"].map(PRIMARY_PLAY_OUTCOME)]
    else:
        a = assoc[assoc["outcome"] == outcome]
    m = a.pivot_table(index="feature", columns="pos_group", values=value)
    q = a.pivot_table(index="feature", columns="pos_group", values="pq" if value == "prho" else "q")
    m = m.reindex(columns=[p for p in POS_ORDER if p in m.columns])
    q = q.reindex_like(m)
    order = m.abs().mean(axis=1).sort_values(ascending=False).index
    m, q = m.loc[order], q.loc[order]
    fig, ax = plt.subplots(figsize=(1.0 * m.shape[1] + 4.2, 0.36 * m.shape[0] + 1.4))
    ax.imshow(m.to_numpy(float), cmap=DIVERGING, vmin=-0.5, vmax=0.5, aspect="auto")
    ax.grid(False)
    ax.set_xticks(range(m.shape[1]), m.columns)
    ax.set_yticks(range(m.shape[0]), [FEATURE_LABELS.get(f, f) for f in m.index])
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = m.iat[i, j]
            if np.isfinite(v):
                sig = np.isfinite(q.iat[i, j]) and q.iat[i, j] < 0.10
                ax.text(j, i, f"{v:+.2f}{'*' if sig else ''}", ha="center", va="center", fontsize=8,
                        color=INK if abs(v) < 0.35 else SURFACE, fontweight="bold" if sig else "normal")
    what = "each position's play-level outcome" if outcome == "primary" else outcome.replace("_", " ")
    ax.set_title(f"Draft-adjusted partial ρ with {what}  (* BH q < 0.10)")
    if outcome == "primary":
        ax.set_xticks(range(m.shape[1]),
                      [f"{p}\n{PRIMARY_PLAY_OUTCOME[p].replace('_', ' ')}" for p in m.columns])
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    return _save(fig, out, "fig4_value_beyond_draft" + ("_play" if outcome == "primary" else ""))


def fig_pass_rush(pr: pd.DataFrame, out: Path) -> Path:
    """Dot-and-whisker: draft-adjusted correlation with pressure rate, EDGE vs interior."""
    _style()
    roles = [r for r in ("EDGE", "IDL") if r in set(pr["role"])]
    feats = list(dict.fromkeys(pr["feature"]))
    fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(feats) + 1.4))
    names = {"EDGE": "Edge rushers (DE/OLB)", "IDL": "Interior (DT/NT)"}
    for k, role in enumerate(roles):
        g = pr[pr["role"] == role].set_index("feature").reindex(feats)
        ys = np.arange(len(feats)) + (k - (len(roles) - 1) / 2) * 0.28
        ax.errorbar(g["prho"], ys, xerr=[g["prho"] - g["plo"], g["phi"] - g["prho"]], fmt="o",
                    color=SERIES[k], ms=6, lw=1.6, capsize=0,
                    label=f"{names[role]}, n≤{int(g['n'].max())}")
    ax.axvline(0, color=AXIS, lw=1)
    ax.set_yticks(range(len(feats)), [FEATURE_LABELS.get(f, f) for f in feats])
    ax.invert_yaxis()
    ax.set(xlabel="Partial ρ with pressure rate, controlling for draft slot (95% bootstrap CI)",
           title="Pass rush: which traits predict pressure beyond draft slot?")
    ax.legend(loc="upper center", bbox_to_anchor=(0.4, -0.13), ncol=2, fontsize=8)
    ax.grid(axis="y", visible=False)
    return _save(fig, out, "fig6_pass_rush")


def fig_increment(inc: pd.DataFrame, out: Path, outcome="snaps_per_season") -> Path:
    _style()
    d = inc[inc["outcome"] == outcome]
    models = list(dict.fromkeys(d["model"]))
    pos = [p for p in POS_ORDER if p in set(d["pos_group"])]
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    w = 0.8 / max(len(models), 1)
    for k, mname in enumerate(models[:3]):
        g = d[d["model"] == mname].set_index("pos_group").reindex(pos)
        xs = np.arange(len(pos)) + (k - (len(models) - 1) / 2) * w
        ax.bar(xs, g["r2"], width=w * 0.92, color=SERIES[k], label=mname)
        ax.errorbar(xs, g["r2"], yerr=g["r2_sd"], fmt="none", ecolor=INK2, lw=1, capsize=2)
    ax.axhline(0, color=AXIS, lw=1)
    ax.set_xticks(range(len(pos)), pos)
    ax.set(ylabel="Out-of-fold R² (rank outcome)",
           title=f"Does tracking add predictive value for {outcome.replace('_', ' ')}?")
    ax.legend(loc="upper left", fontsize=8, ncol=3)
    ax.grid(axis="x", visible=False)
    return _save(fig, out, "fig5_incremental_value")


In [ ]:
%%writefile {SRC}/combine2field/synthetic.py
"""SYNTHETIC data with the same schema as the Big Data Bowl 2027 files.

Used only for unit tests, CI and the app's demo mode. None of these numbers are
real players or real measurements. The competition data (CC BY-NC 4.0) is never
stored in this repository.

Each fake player has a latent "engine" (top speed S0, acceleration time
constant tau, braking capacity) and a latent "expression" fraction that controls
how much of that engine shows up in games. Career outcomes depend on both, so the
pipeline has a known signal to recover.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from . import config

POS_PARAMS = {
    #       S0 mean (yd/s), tau mean (s), weight (lb), height (in)
    "WR": (10.2, 1.05, 200, 72),
    "DB": (10.1, 1.05, 195, 71),
    "TE": (9.5, 1.15, 250, 76),
    "DL": (8.9, 1.20, 295, 75),
    "OL": (8.2, 1.30, 315, 77),
}
NFL_POS = {"WR": ["WR"], "DB": ["CB", "SS", "FS"], "TE": ["TE"], "DL": ["DE", "DT", "OLB"], "OL": ["T", "G", "C"]}
T0 = pd.Timestamp("2025-03-01 10:00:00", tz="UTC")


def _iso(seconds: np.ndarray) -> list[str]:
    ts = T0 + pd.to_timedelta(seconds, unit="s")
    return [t.strftime("%Y-%m-%dT%H:%M:%S.%f") for t in ts]


def _sprint(s0, tau, dist=40.0, pre=0.8, post=1.5, rng=None):
    """Mono-exponential sprint over `dist` yards, plus standing and slowing phases."""
    t_run = np.arange(0, 8, config.DT)
    v = s0 * (1 - np.exp(-t_run / tau))
    x = s0 * (t_run - tau * (1 - np.exp(-t_run / tau)))
    end = int(np.searchsorted(x, dist)) + 1
    v, x = v[:end], x[:end]
    # slow down after the line
    n_post = int(post / config.DT)
    v_post = np.maximum(v[-1] - 6.0 * config.DT * np.arange(1, n_post + 1), 0.5)
    x_post = x[-1] + np.cumsum(v_post) * config.DT
    n_pre = int(pre / config.DT)
    v_all = np.concatenate([np.zeros(n_pre), v, v_post])
    x_all = np.concatenate([np.zeros(n_pre), x, x_post])
    noise = rng.normal(0, 0.03, x_all.size) if rng is not None else 0
    return x_all + noise, v_all


def _waypoint_run(points, s0, a0, dmax, rng):
    """Point-to-point run with an acceleration cap that falls with speed and a braking cap."""
    pos = np.array(points[0], float)
    vel = np.zeros(2)
    out = [pos.copy()]
    for target in points[1:]:
        target = np.asarray(target, float)
        for _ in range(200):
            d = target - pos
            dist = np.linalg.norm(d)
            if dist < 0.15:
                break
            u = d / dist
            speed = np.linalg.norm(vel)
            v_des = min(s0, np.sqrt(2 * dmax * max(dist - 0.1, 0)))
            v_cur_along = vel @ u
            if v_des > v_cur_along:
                acc = a0 * max(1 - speed / s0, 0.05)
            else:
                acc = -dmax
            new_speed_along = np.clip(v_cur_along + acc * config.DT, 0, s0)
            # lateral velocity is damped by braking capacity
            lat = vel - v_cur_along * u
            lat_mag = np.linalg.norm(lat)
            lat = lat * max(lat_mag - dmax * config.DT, 0) / lat_mag if lat_mag > 1e-9 else lat
            vel = new_speed_along * u + lat
            pos = pos + vel * config.DT
            out.append(pos.copy())
    out = np.array(out)
    return out + rng.normal(0, 0.02, out.shape)


def _speed_acc(xy: np.ndarray):
    v = np.gradient(xy, config.DT, axis=0)
    s = np.linalg.norm(v, axis=1)
    a = np.abs(np.gradient(s, config.DT))
    return s, a


def generate(n_per_pos: int = 24, plays_per_player: int = 60, seed: int = 7) -> dict[str, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    players, results, career, ct_rows, gt_rows, pp_rows = [], [], [], [], [], []
    nfl_id = 60000
    play_counter = 0
    games = pd.DataFrame({
        "game_id": np.arange(2023090700, 2023090700 + 30),
        "season": np.repeat([2023, 2024, 2025], 10),
        "week": np.tile(np.arange(1, 11), 3),
        "game_key": np.arange(30),
        "home_team_abbr": "HOM", "visitor_team_abbr": "VIS", "season_type": "REG",
    })
    event_seq = 0
    for pos, (s0_mu, tau_mu, w_mu, h_mu) in POS_PARAMS.items():
        for _ in range(n_per_pos):
            nfl_id += 1
            draft_year = int(rng.choice([2023, 2024, 2025]))
            s0 = rng.normal(s0_mu, 0.35)
            tau = max(rng.normal(tau_mu, 0.08), 0.75)
            a0 = s0 / tau
            dmax = max(rng.normal(7.0, 0.9), 4.0)
            expression = np.clip(rng.beta(6, 2.5), 0.45, 0.99)
            talent = 0.6 * (s0 - s0_mu) / 0.35 + 0.4 * (expression - 0.7) / 0.12 + rng.normal(0, 0.7)
            pick_score = 0.5 * (s0 - s0_mu) / 0.35 + rng.normal(0, 1.0)  # teams see the engine only
            pick = int(np.clip(130 - 55 * pick_score, 1, 300))
            drafted = pick <= 257
            seasons = 2025 - draft_year + 1
            snaps = max(0, int(seasons * (350 + 260 * talent + rng.normal(0, 120))))
            off = pos in ("WR", "TE", "OL")
            players.append({
                "nfl_id": nfl_id, "display_name": f"Synthetic {pos} {nfl_id}",
                "birth_date": "2001-01-01", "college_name": "Synthetic U", "college_conference": "SYN",
                "draft_year": draft_year,
                "draft_round": (pick - 1) // 32 + 1 if drafted else np.nan,
                "draft_pick_within_round": (pick - 1) % 32 + 1 if drafted else np.nan,
                "draft_overall_pick": pick if drafted else np.nan,
                "nfl_position": rng.choice(NFL_POS[pos]),
            })
            forty_t = _sprint(s0, tau)[1]
            x40, _ = _sprint(s0, tau)
            forty = (np.searchsorted(x40[8:], 40) * config.DT) + rng.normal(0.05, 0.02)
            ten = (np.searchsorted(x40[8:], 10) * config.DT) + rng.normal(0.05, 0.02)
            results.append({
                "nfl_id": nfl_id, "draft_year": draft_year, "combine_position": pos,
                "combine_height": rng.normal(h_mu, 1.5), "combine_weight": rng.normal(w_mu, 10),
                "hand_size": rng.normal(9.6, 0.4), "arm_length": rng.normal(32.5, 1.2),
                "wing_span": rng.normal(78, 2.5), "forty": round(forty, 2), "ten_yd_split": round(ten, 2),
                "vertical": rng.normal(33, 3), "broad_jump": rng.normal(118, 6),
                "three_cone": rng.normal(7.1, 0.2), "short_shuttle": rng.normal(4.4, 0.12),
                "bench_reps": rng.normal(20, 5),
                "ngs_athleticism_score": int(np.clip(rng.normal(72, 10), 40, 99)),
                "ngs_college_production_score": int(np.clip(rng.normal(70, 10), 40, 99)),
                "ngs_final_score": int(np.clip(rng.normal(70, 10), 40, 99)),
            })
            career.append({
                "nfl_id": nfl_id,
                "career_offensive_snaps": snaps if off else 0,
                "career_defensive_snaps": 0 if off else snaps,
                "career_special_teams_snaps": int(rng.integers(0, 300)),
                "career_games_active": int(min(17 * seasons, max(0, 8 * seasons + 4 * talent * seasons))),
                "career_games_started": int(min(17 * seasons, max(0, 4 * seasons + 4 * talent * seasons))),
                "pro_bowl_original_ballot": int(talent > 2.0),
                "ap_all_pro_1st_team": int(talent > 2.6),
                "ap_all_pro_2nd_team": 0,
            })
            # ---- combine tracking: two 40s and two of each COD drill
            t_clock = float(event_seq * 100)
            for attempt in (1, 2):
                event_seq += 1
                x, _ = _sprint(s0 * rng.normal(1, 0.01), tau * rng.normal(1, 0.03), rng=rng)
                xy = np.column_stack([20 + x, 25 + rng.normal(0, 0.03, x.size)])
                ct_rows.append(_ct_frame(nfl_id, draft_year, config.DRILL_FORTY, "forty_yard_dash",
                                         attempt, event_seq, t_clock, xy))
                t_clock += 30
            shuttle = [(0, 0), (0, 5), (0, -5), (0, 0)]
            cone = [(0, 0), (5, 0), (0, 0), (5, 0), (5, 5), (0, 5.2), (5, 5), (5.2, 0), (0, 0)]
            for drill, dtype, pts in ((config.DRILL_SHUTTLE, "short_shuttle", shuttle),
                                      (config.DRILL_THREE_CONE, "three_cone_drill", cone)):
                for attempt in (1, 2):
                    event_seq += 1
                    xy = _waypoint_run(pts, s0 * 0.85, a0, dmax * rng.normal(1, 0.03), rng)
                    xy = np.vstack([np.repeat(xy[:1], 6, axis=0), xy]) + [40, 20]
                    ct_rows.append(_ct_frame(nfl_id, draft_year, drill, dtype, attempt, event_seq, t_clock, xy))
                    t_clock += 30
            # ---- game tracking: sprints at intensity <= expression
            season_games = games[games["season"] >= draft_year]["game_id"].to_numpy()
            for k in range(plays_per_player):
                play_counter += 1
                gid = int(rng.choice(season_games))
                intensity = expression * rng.uniform(0.55, 1.0)
                v0 = rng.uniform(0, 2.5)
                t = np.arange(0, 4.5, config.DT)
                vt = s0 * intensity
                v = vt - (vt - v0) * np.exp(-t / (tau * rng.uniform(1.0, 1.25)))
                v[t > 3.3] = np.maximum(v[t > 3.3][0] - 5 * (t[t > 3.3] - 3.3), 0)
                ang = rng.uniform(0, 2 * np.pi)
                d = np.cumsum(v) * config.DT
                xy = np.column_stack([60 + d * np.cos(ang), 26 + d * np.sin(ang)])
                s, a = _speed_acc(xy)
                gt_rows.append(pd.DataFrame({
                    "game_id": gid, "play_id": play_counter, "nfl_id": nfl_id,
                    "time": _iso(10_000 + play_counter * 10 + t), "x": xy[:, 0], "y": xy[:, 1],
                    "s": s, "a": a, "dis": s * config.DT, "o": 0.0, "dir": np.degrees(ang) % 360,
                    "event": ["ball_snap"] + [np.nan] * (t.size - 1),
                }))
                pp_rows.append(_pp_row(pos, gid, play_counter, nfl_id, talent, rng))
    data = {
        "players": pd.DataFrame(players),
        "combine_results": pd.DataFrame(results),
        "player_career_successes": pd.DataFrame(career),
        "games": games,
        "combine_tracking": pd.concat(ct_rows, ignore_index=True),
        "player_play": pd.DataFrame(pp_rows),
    }
    gt = pd.concat(gt_rows, ignore_index=True)
    season_of = games.set_index("game_id")["season"]
    for season in config.SEASONS:
        data[f"game_tracking_{season}"] = gt[gt["game_id"].map(season_of) == season].reset_index(drop=True)
    return data


def _ct_frame(nfl_id, draft_year, drill, dtype, attempt, event_seq, t_clock, xy):
    s, a = _speed_acc(xy)
    t = t_clock + np.arange(len(xy)) * config.DT
    return pd.DataFrame({
        "nfl_id": nfl_id, "draft_year": draft_year, "drill_name": drill.upper(),
        "drill_type": dtype.upper(), "entity_type": "PLAYER", "attempt": attempt, "event_id": f"SYN_{event_seq:05d}",
        "time": _iso(t), "x": xy[:, 0], "y": xy[:, 1], "s": s, "a": a,
        "dis": s * config.DT, "dir": 90.0,
    })


def _pp_row(pos, gid, play_id, nfl_id, talent, rng):
    row = {"game_id": gid, "play_id": play_id, "nfl_id": nfl_id, "team_abbr": "SYN",
           "route_ran": np.nan, "separation_at_pass_forward": np.nan, "target": False,
           "pressure_allowed": np.nan, "time_to_pressure": np.nan, "player_get_off": np.nan,
           "sack": 0, "tackle": np.nan, "assist": np.nan, "pass_rushers_encountered": np.nan}
    if pos in ("WR", "TE"):
        row["route_ran"] = rng.choice(["go", "hitch", "slant", "out", "post"])
        row["separation_at_pass_forward"] = max(0.3, rng.normal(3.0 + 0.35 * talent, 1.2))
        row["target"] = bool(rng.random() < 0.18 + 0.03 * talent)
    elif pos == "OL":
        row["pressure_allowed"] = bool(rng.random() < max(0.02, 0.07 - 0.015 * talent))
        row["pass_rushers_encountered"] = 1
    elif pos == "DL":
        pressured = rng.random() < max(0.02, 0.10 + 0.025 * talent)
        row["time_to_pressure"] = rng.uniform(1.8, 3.5) if pressured else np.nan
        row["player_get_off"] = max(0.4, rng.normal(0.95 - 0.04 * talent, 0.1))
    else:
        if rng.random() < 0.08 + 0.01 * talent:  # real data: blank unless involved
            solo = rng.random() < 0.6
            row["tackle"], row["assist"] = (1, 0) if solo else (0, 1)
    return row


def write(out_dir: Path, **kw) -> Path:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, df in generate(**kw).items():
        df.to_csv(out_dir / f"{name}.csv", index=False)
    (out_dir / "SYNTHETIC_DATA_README.txt").write_text(
        "These files are SYNTHETIC. They copy the schema of the Big Data Bowl 2027 data but "
        "contain no real measurements.\n"
    )
    return out_dir


In [ ]:
%%writefile {SRC}/combine2field/pipeline.py
"""End-to-end pipeline: raw competition CSVs -> features, tables, figures.

    combine2field                         # uses data/raw or the Kaggle mount
    combine2field --synthetic             # demo on generated data
    combine2field --data-dir D --work-dir W --repeats 50
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from . import combine_features as cf
from . import config, game_profile, io, outcomes, report, synthetic, translation


def log(msg: str, t0: float) -> None:
    print(f"[{time.time() - t0:7.1f}s] {msg}", flush=True)


def run(data_dir: Path, work: Path, n_repeats: int = 20, make_figures: bool = True,
        exposure: int | None = config.EXPOSURE_PLAYS) -> dict:
    t0 = time.time()
    tables = work / "tables"
    figs = work / "figures"
    tables.mkdir(parents=True, exist_ok=True)

    base = io.player_base(data_dir)
    base = base[base["pos_group"].isin(config.POSITION_GROUPS)]
    ids = set(base["nfl_id"])
    log(f"players: {len(base)}", t0)

    ct = io.combine_tracking(data_dir, work)
    attempts = cf.attempt_features(ct)
    engine = cf.player_engine(attempts)
    rel = translation.reliability(cf.retest_pairs(attempts))
    log(f"combine attempts: {len(attempts)}, players with engine: {len(engine)}", t0)

    expression, bins, plays = game_profile.build(ids, data_dir, work, plays_per_player=exposure)
    log(f"game expression for {expression['game_A0'].notna().sum()} players "
        f"({exposure or 'all'} plays each)", t0)

    car = outcomes.career_outcomes(base)
    pp = io.player_play(data_dir, outcomes.PLAY_COLUMNS)
    play_out = outcomes.play_outcomes(pp)
    df = translation.assemble(base, engine, expression, car, play_out)
    log("assembled player table", t0)

    trans = translation.translation_table(df)
    assoc = translation.association_table(df)
    inc = translation.incremental_value(df, n_repeats=n_repeats)
    pass_rush = translation.pass_rush_table(df)
    log("statistics done", t0)

    df.to_parquet(tables / "players.parquet", index=False)
    bins.to_parquet(tables / "game_as_bins.parquet", index=False)
    attempts.to_parquet(tables / "combine_attempts.parquet", index=False)
    for name, t in {"reliability": rel, "translation": trans, "associations": assoc,
                    "incremental_value": inc, "pass_rush": pass_rush}.items():
        t.to_csv(tables / f"{name}.csv", index=False)

    if make_figures:
        ex = df.dropna(subset=["game_A0", "eng_A0"])
        if len(ex):
            pick = ex[ex["pos_group"] == "WR"] if (ex["pos_group"] == "WR").any() else ex
            report.fig_method(ct, bins, pick.sort_values("game_n_plays").iloc[-1], figs)
        report.fig_reliability(rel, figs)
        report.fig_translation(df, figs)
        if not assoc.empty:
            report.fig_heatmap(assoc, figs)
            report.fig_heatmap(assoc, figs, outcome="primary")
        if not pass_rush.empty and pass_rush["prho"].notna().any():
            report.fig_pass_rush(pass_rush, figs)
        if not inc.empty:
            report.fig_increment(inc, figs)
        log(f"figures -> {figs}", t0)

    summary = summarise(df, rel, trans, assoc, inc)
    if not pass_rush.empty:
        summary["pass_rush"] = pass_rush[["role", "feature", "n", "prho", "plo", "phi", "pp", "pq"]] \
            .round(3).to_dict("records")
    (work / "results.json").write_text(json.dumps(summary, indent=2, default=float))
    log(f"done -> {work}", t0)
    return summary


def summarise(df, rel, trans, assoc, inc) -> dict:
    s = {"n_players": int(len(df)),
         "n_by_position": df["pos_group"].value_counts().to_dict(),
         "n_with_engine": int(df["eng_S0"].notna().sum()),
         "n_with_expression": int(df["game_A0"].notna().sum())}
    s["exposure_check_rho_nplays"] = {
        c: round(float(df[[c, "game_n_plays_total"]].corr(method="spearman").iloc[0, 1]), 3)
        for c in ("game_vmax", "game_S0", "game_A0") if c in df and "game_n_plays_total" in df}
    s["median_speed_expression"] = df.groupby("pos_group")["expr_speed"].median().round(3).to_dict()
    s["top_reliability"] = rel.head(5)[["drill_name", "metric", "icc"]].to_dict("records")
    if not assoc.empty:
        sig = assoc[(assoc["pq"] < 0.10)].sort_values("pq")
        s["significant_beyond_draft"] = sig[["pos_group", "outcome", "feature", "prho", "pq"]] \
            .head(15).to_dict("records")
    if not inc.empty:
        piv = inc.pivot_table(index=["pos_group", "outcome"], columns="model", values="r2")
        s["incremental_r2"] = {f"{p}|{o}": {k: (None if np.isnan(v) else round(v, 3)) for k, v in row.items()}
                               for (p, o), row in piv.iterrows()}
    return s


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data-dir", type=Path, default=None)
    ap.add_argument("--work-dir", type=Path, default=None)
    ap.add_argument("--synthetic", action="store_true", help="generate and analyse SYNTHETIC data")
    ap.add_argument("--repeats", type=int, default=20, help="CV repeats for incremental value")
    ap.add_argument("--no-figures", action="store_true")
    ap.add_argument("--exposure", type=int, default=config.EXPOSURE_PLAYS,
                    help="random plays per player for in-game metrics (0 = use all plays)")
    a = ap.parse_args(argv)
    work = a.work_dir or config.work_dir()
    data_dir = a.data_dir or config.data_dir()
    if a.synthetic:
        work = a.work_dir or (config.ROOT / "outputs" / "synthetic")
        data_dir = synthetic.write(work / "raw", n_per_pos=30, plays_per_player=60)
    pd.set_option("display.width", 160)
    exposure = a.exposure or None
    if a.synthetic:
        exposure = min(exposure or 40, 40)
    summary = run(data_dir, work, a.repeats, not a.no_figures, exposure)
    print(json.dumps({k: summary[k] for k in ("n_players", "n_with_engine", "n_with_expression",
                                              "median_speed_expression")}, indent=2, default=float))


if __name__ == "__main__":
    main()


## 1 · Run the pipeline
Reads the competition files from `/kaggle/input`. Derived tables are written to a temp directory (not published); figures are copied to `/kaggle/working/figures`.

In [ ]:
import os, shutil, json
from pathlib import Path
from combine2field import config, pipeline
DATA = config.data_dir()
EXPOSURE = config.EXPOSURE_PLAYS
if not (DATA / 'players.csv').exists():
    if config.on_kaggle():
        found = sorted(str(p) for p in Path('/kaggle/input').rglob('*.csv'))[:20]
        raise FileNotFoundError(
            'Competition data not attached. In the right panel click "Add Input" -> '
            'Competitions -> NFL Big Data Bowl 2027, then run again. CSVs found: ' + str(found))
    print('Competition data not found -> running on SYNTHETIC data (demo only)')
    from combine2field import synthetic
    DATA = synthetic.write(Path(TMP) / 'c2f_raw', n_per_pos=30, plays_per_player=60)
    EXPOSURE = 40
print('data:', DATA)
WORK = Path(TMP) / 'c2f'
summary = pipeline.run(DATA, WORK, n_repeats=int(os.environ.get('C2F_REPEATS', 50)), exposure=EXPOSURE)
PUBLIC = Path('/kaggle/working') if config.on_kaggle() else Path(TMP) / 'c2f_public'
shutil.copytree(WORK / 'figures', PUBLIC / 'figures', dirs_exist_ok=True)
print(json.dumps({k: summary[k] for k in ('n_players','n_with_engine','n_with_expression')}, indent=2))

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.precision', 3)
T = WORK / 'tables'; F = WORK / 'figures'
players = pd.read_parquet(T / 'players.parquet')
rel = pd.read_csv(T / 'reliability.csv'); trans = pd.read_csv(T / 'translation.csv')
assoc = pd.read_csv(T / 'associations.csv'); inc = pd.read_csv(T / 'incremental_value.csv')

## 2 · One model, two environments
Left: the 40 is fitted with $v(t)=S_0\,(1-e^{-(t-t_0)/\tau})$, which implies the linear profile $a(v)=A_0(1-v/S_0)$ with $A_0=S_0/\tau$. Right: every game frame at ≥3 yd/s is binned by speed; the two highest tangential accelerations per 0.2 yd/s bin are kept and a robust line gives the in-game $A_0$ and $S_0$ (Morin et al., 2021).

In [ ]:
display(Image(F / 'fig1_method.png'))

## 3 · Reliability: which tracking metrics are traits?
A metric that does not repeat between attempt 1 and attempt 2 cannot be a trait. ICC(3,1) ≥ 0.75 is conventionally *good*.

In [ ]:
display(Image(F / 'fig2_reliability.png'))
rel.head(15)

## 4 · Translation: how much of the engine is expressed?

In [ ]:
display(Image(F / 'fig3_translation.png'))
players.groupby('pos_group')[['expr_speed','expr_accel','expr_brake']].median()

In [ ]:
trans.pivot_table(index=['combine','game'], columns='pos_group', values='rho')

## 5 · Value beyond draft slot
Spearman ρ between each metric and snaps per season, after removing the rank-linear effect of log(draft pick). Signs are oriented so positive = better athlete → more playing time. Benjamini-Hochberg q-values control false discoveries within each position.

In [ ]:
display(Image(F / 'fig4_value_beyond_draft.png'))
display(Image(F / 'fig4_value_beyond_draft_play.png'))
assoc[assoc['pq'] < 0.10].sort_values('pq')[['pos_group','outcome','feature','n','rho','prho','pq']]

### 5b · Pass rush: edge vs. interior
Edge rushers are lighter, faster and pressure more often, so a pooled DL correlation partly measures role. Within each role: draft-adjusted partial ρ with pressure rate, 95% bootstrap CI.

In [ ]:
display(Image(F / 'fig6_pass_rush.png'))
pd.read_csv(T / 'pass_rush.csv')[['role','feature','n','prho','plo','phi','pq']]

## 6 · Do tracking metrics add out-of-fold predictive value?
Nested ridge models, 5-fold CV repeated 50 times, rank-transformed outcome: draft slot → + stopwatch Combine → + tracking engine & expression.

In [ ]:
display(Image(F / 'fig5_incremental_value.png'))
inc.pivot_table(index=['pos_group','outcome'], columns='model', values='r2')

## 7 · Limitations
* Sample sizes are about 100 players per group, and the 2025 class has one season of outcomes.
* Game expression depends on role: a slot WR rarely reaches top speed, so expression mixes ability with usage. We report it separately from the engine rather than as a single verdict.
* Snaps and starts measure opportunity as well as quality; play-level outcomes are added where the data supports them.
* All associations are observational.